In [6]:
from data_utils import HarmonicGraphDataset, harmonic_graph_collate_fn
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
import torch
from torch.utils.data import DataLoader

from models_graph import HarmonicGraphEncoder
from generate_utils import load_AttnFiLMSEModel

import pickle

In [7]:
# load the dataset and the tokenizer
with open("data/gjt_train.pkl", "rb") as f:
    gjt_train = pickle.load(f)

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graph_model = HarmonicGraphEncoder()
graph_model.to(device)

# load the model
model = load_AttnFiLMSEModel(
    tokenizer=tokenizer,
    guidance_dim=graph_model.output_dim
)

In [9]:
dataset = HarmonicGraphDataset(gjt_train, tokenizer, model)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=harmonic_graph_collate_fn)

In [10]:
batch = next(iter(dataloader))
print(batch.keys())

dict_keys(['pianoroll', 'real_harmony_ids', 'recomposed_harmony_ids', 'random_harmony_ids', 'real_graph', 'recomposed_graph', 'random_graph'])


In [ ]:
logits = model(
    batch['pianoroll'].to(device),
    batch['real_harmony_ids'].to(device), # TODO: masked version
    batch['real_graph'].to(device)
)
# TODO: when training with graph guidance, we can train for all graph segment
# masked tokens in parallel